<a href="https://colab.research.google.com/github/MGentieu/dl_project/blob/main/starters/ts-project-starter/ts-project/notebooks/mlp_california_housing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regression Starter — Colab

> Author: Gouesse Sixtine & Gentieu Martin

Ce notebook est un projet de bout en bout pour de la **régression tabulaire** sur le dataset **California Housing** avec un **MLP PyTorch**.

# M1 — Problem Scoping & Data Validation

### Problem Statement
L’objectif de ce projet est d’entraîner un modèle **MLP (Multi-Layer Perceptron)** pour
**prédire la valeur médiane des maisons** (en dollars) sur le dataset **California Housing**.

- **Entrée du modèle :** vecteur de features tabulaires (longitude, latitude, rooms, income, etc.)
- **Sortie :** une valeur réelle (prix médian de la maison dans le district)

## Model Inputs

Les features d’entrée (colonnes) :

- `longitude`
- `latitude`
- `housing_median_age`
- `total_rooms`
- `total_bedrooms`
- `population`
- `households`
- `median_income`

La cible (target) :

- `median_house_value`

## Model Outputs

- Une **valeur réelle** : prédiction du `median_house_value`.

### Evaluation Metrics
- **MSE**
- **MAE**
- **R²**
- **plot of predictions vs ground truth**

## Data Card – California Housing Dataset

**Nom du dataset :** California Housing

**Source :** Version Kaggle du dataset "California Housing Prices" (Cam Nugent).

**Tâche :** Régression (prédire un prix à partir de variables tabulaires).

**Format brut :**

- Fichier CSV unique `housing.csv`,
- Chaque ligne correspond à un district en Californie,
- Chaque colonne correspond à une caractéristique agrégée (moyenne, médiane).

**Pré-processing appliqué dans ce notebook :**

- Séparation en splits **train / val / test** : 70% / 15% / 15%,
- Standardisation (z-score) des features + target,
- Construction de DataLoaders PyTorch pour MLP.

**Impact & risques :**

- Modèle entraîné sur des données historiques, potentiellement obsolètes par rapport au marché actuel,
- Risque d’interprétation incorrecte si on applique ce modèle dans un cadre réel sans recalibrage.


## Ouverture du dataset et installation

### Connexion à github

Pour github : à exécuter dans le terminal de colab

git clone https://github.com/MGentieu/dl_project.git

### Step 0 — Confirm the GPU is ready
Run the next cell. You should see GPU name + memory. If you see `nvidia-smi unavailable`, switch the runtime to GPU and rerun this cell.

In [ ]:
!nvidia-smi || echo "nvidia-smi unavailable (CPU runtime)"


### Step 1 — Point the notebook at the project folder
This cell switches into the `ts-project` directory.
If you get a `FileNotFoundError`, confirm where you uploaded/cloned the folder, adjust the path, and rerun.

In [ ]:
import os
import sys
from pathlib import Path

# The user has provided the explicit path to the project root.
PROJECT_ROOT = Path("/content/dl_project/starters/ts-project-starter/ts-project")

print(f"Environment: Colab/Kaggle (remote server), using provided PROJECT_ROOT")

# Validate structure
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(f"Missing src/ directory at {PROJECT_ROOT}")

# Setup Python path
os.chdir(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {Path.cwd()}")

# Required paths inside the project
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CONFIG_DIR = PROJECT_ROOT / "configs"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"

# Create dirs if missing
for d in [DATA_DIR, OUTPUT_DIR, CONFIG_DIR, NOTEBOOK_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project folders validated:")
print(f"  DATA_DIR    = {DATA_DIR}")
print(f"  OUTPUT_DIR  = {OUTPUT_DIR}")
print(f"  CONFIG_DIR  = {CONFIG_DIR}")
print(f"  NOTEBOOK_DIR= {NOTEBOOK_DIR}")


### Step 2 — Install the project requirements
Installs PyTorch + supporting libraries from `requirements.txt`. Expect quite a bit of output. If the install fails, rerun the cell before continuing.

In [ ]:
!pip install torch
!pip install numpy
!pip install pandas
!pip install matplotlib
!pip install tqdm
!pip install pyyaml
!pip install kagglehub
!pip install scikit-learn


In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")


In [ ]:
# ============================================================
# Imports
# ============================================================
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from tqdm import tqdm
import kagglehub
import yaml

print("Imports loaded successfully.")

# ============================================================
# Reproducibility
# ============================================================

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print("Torch version:", torch.__version__)


### Step 3 - Download and view samples

On télécharge maintenant le dataset via **KaggleHub** et on inspecte sa structure.

In [ ]:
print("Downloading California Housing dataset via KaggleHub...")
ds_path = kagglehub.dataset_download("camnugent/california-housing-prices")
ds_path = Path(ds_path)

print("Dataset downloaded at:", ds_path)

# Create dataset folder inside the project
CAL_DIR = DATA_DIR / "california_housing"
CAL_DIR.mkdir(parents=True, exist_ok=True)

# Copy CSV into project structure
raw_csv = ds_path / "housing.csv"
target_csv = CAL_DIR / "housing.csv"

target_csv.write_bytes(raw_csv.read_bytes())
print("Copied housing.csv →", target_csv)


In [ ]:
# Quick inspection
df = pd.read_csv(target_csv)
print(df.head())
print(df.describe())

### Step 4 - Build MLP Dataset (Train/Val/Test)

1. Convertir le dataset au **format MLP**,
2. Créer un fichier `california_housing.yaml`,
3. Entraîner un modèle **MLP** baseline.

Le split sera :
- 70% train
- 15% val
- 15% test


Creation fichier YAML

In [ ]:
# ============================================
# STEP 3 — CREATE YAML CONFIG
# ============================================

config = {
    "task": "regression_tabular",
    "seed": SEED,
    "output_dir": str(OUTPUT_DIR),
    "data": {
        "csv_path": str(target_csv),
        "target_col": "median_house_value",
        "feature_cols": None,
        "val_split": 0.15,
        "test_split": 0.15,
        "shuffle": True,
        "standardize": True,
    },
    "model": {
        "type": "mlp",
        "hidden_dims": [256, 128],
        "dropout": 0.1,
        "activation": "relu",
    },
    "train": {
        "epochs": 30,
        "batch_size": 128,
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "optimizer": "adamw",
        "momentum": 0.9,
        "scheduler": "cosine",
        "t_max": 30,
    },
    "early_stopping": {
        "patience": 5,
        "min_delta": 0.0,
    },
}

YAML_PATH = CONFIG_DIR / "california_housing.yaml"
with open(YAML_PATH, "w") as f:
    yaml.dump(config, f)

print("Generated config file:", YAML_PATH)
print(open(YAML_PATH).read())

Split Train / Val / Test

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data_cfg = cfg["data"]

df = pd.read_csv(data_cfg["csv_path"])

target_col = data_cfg["target_col"]
feature_cols = data_cfg["feature_cols"]

if feature_cols is None:
    # all numeric columns except target
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols.remove(target_col)
    feature_cols = numeric_cols

print("Features:", feature_cols)
print("Target:", target_col)

X = df[feature_cols].values
y = df[target_col].values.reshape(-1, 1)

val_split = data_cfg["val_split"]
test_split = data_cfg["test_split"]

# First split off test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=test_split, random_state=SEED, shuffle=data_cfg["shuffle"]
)

# Then split train / val
val_ratio_relative = val_split / (1.0 - test_split)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=val_ratio_relative,
    random_state=SEED,
    shuffle=data_cfg["shuffle"]
)

print("Train shape:", X_train.shape)
print("Val   shape:", X_val.shape)
print("Test  shape:", X_test.shape)

Standardisation (features + target)

In [ ]:
standardize = data_cfg.get("standardize", True)

if standardize:
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_train = x_scaler.fit_transform(X_train)
    X_val = x_scaler.transform(X_val)
    X_test = x_scaler.transform(X_test)

    y_train = y_scaler.fit_transform(y_train)
    y_val = y_scaler.transform(y_val)
    y_test = y_scaler.transform(y_test)

    print("Standardization applied.")
else:
    x_scaler = None
    y_scaler = None
    print("No standardization applied.")

Construction des DataLoaders PyTorch

In [ ]:
rain_ds = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32),
)
val_ds = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32),
)
test_ds = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32),
)

batch_size = cfg["train"]["batch_size"]

train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

len(train_dl), len(val_dl), len(test_dl)

# M2 — Baseline Model Implementation

Nous utilisons MLP comme modèle baseline.  
Avant tout entraînement, nous validons que :

1. Le modèle se charge correctement  
2. Un batch peut passer dans le modèle sans erreur  
3. L’entraînement démarre correctement  

### On importe le baseline model

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout=0.0, activation="relu"):
        super().__init__()
        layers = []
        prev_dim = input_dim

        if activation == "relu":
            act_layer = nn.ReLU
        elif activation == "tanh":
            act_layer = nn.Tanh
        else:
            raise ValueError(f"Unknown activation: {activation}")

        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(act_layer())
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            prev_dim = h

        layers.append(nn.Linear(prev_dim, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

model_cfg = cfg["model"]
input_dim = len(feature_cols)

model = MLP(
    input_dim=input_dim,
    hidden_dims=model_cfg["hidden_dims"],
    dropout=model_cfg["dropout"],
    activation=model_cfg["activation"],
).to(device)

print(model)

### Sanity check

In [ ]:
xb, yb = next(iter(train_dl))
xb = xb.to(device)
with torch.no_grad():
    pred = model(xb)

print("Input batch shape:", xb.shape)
print("Predictions shape:", pred.shape)
print("Sanity check OK.")

### On fait un training initial de 5 epochs

# M3 — Optimization & Regularization

On entraîne un MLP baseline et on mesure :

- Loss (MSE sur données standardisées),
- MSE / MAE / R² sur les données **rescalées** dans l’échelle originale,
- Plot Predicted vs Ground Truth.


## 1. Review smoke-test output
- Confirm the previous cell printed a JSON block with the loss, batch size, and input shape.
- `outputs/smoke_metrics.json` should now exist. If the smoke test failed because data is missing, create a small CSV and rerun it before proceeding.
- Ready for full training? Continue to Section 2.

## 2. Full training run (optional)
Train/evaluate on the configuration you choose. Default `reg_tabular_mlp.yaml` expects a CSV at `data/train.csv` with a numeric `target` column.

**Before running:**
1. Ensure your CSV paths in the config exist (or switch configs to the time-series variant).
2. Open the config if you want to tweak epochs, batch size, learning rate, etc.
3. Confirm the runtime still shows a GPU connection.

In [ ]:
# Train the model defined in configs/reg_tabular_mlp.yaml (tabular regression).
!python src/train.py --config configs/reg_tabular_mlp.yaml


In [ ]:
# Evaluate the saved checkpoint on the validation/test splits.
!python src/evaluate.py --config configs/reg_tabular_mlp.yaml --ckpt outputs/best.pt


### Step 4 — What should I see now?
- `outputs/best.pt`: trained weights + scaler metadata.
- `outputs/log.csv`: epoch-by-epoch loss and regression metrics.
- `outputs/metrics.json`: best validation MSE.
- `outputs/eval.json`: validation (and, if configured, test) MSE/MAE/R².
If any files are missing, scroll up for errors in the training/evaluation cells.

## 3. Switch configurations
- To run the time-series variant, swap the config paths in the training/evaluation cells for `configs/reg_timeseries_lstm.yaml`.
- Update the CSV paths (`data.csv_path`) to point at your series file (must include a timestamp column).
- Re-run the smoke test and full run to validate the new setup.

Keep the same order—GPU check → install → smoke test → train → evaluate—for consistent results.